In [ ]:
import logging
import os
import pickle
import random
import sys
from os.path import join

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

# from hyperopt import fmin, tpe, hp, Trials, rand
import xgboost as xgb
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import KFold

sys.path.append("./additional_code")
# from data_preprocessing import *
CURRENT_DIR = os.getcwd()
print(CURRENT_DIR)
our_data = CURRENT_DIR + "/../data/our_data/"

/home/hanxd/Repositories/ESP/our_codes


In [ ]:
slice0data = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "slice0data.pkl")
)
slice1data = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "slice1data.pkl")
)
slice2data = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "slice2data.pkl")
)
slice3data = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "slice3data.pkl")
)
slice4data = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "slice4data.pkl")
)

p450plant = pd.concat(
    [slice0data, slice1data, slice2data, slice3data, slice4data], ignore_index=True
)

In [ ]:
def create_input_and_output_data(df):
    X = ()
    y = ()
    for ind in df.index:
        emb = df["ESM1b"][ind]
        ecfp = np.array(list(df["ECFP"][ind])).astype(int)

        X = X + (np.concatenate([ecfp, emb]),)
        y = y + (df["Binding"][ind],)

    return (X, y)


feature_names = ["ECFP_" + str(i) for i in range(1024)]
feature_names = feature_names + ["ESM1b_" + str(i) for i in range(1280)]

data_X, data_y = create_input_and_output_data(df=p450plant)

In [ ]:
bst = pd.read_pickle(
    os.path.join(CURRENT_DIR, "..", "data", "our_data", "deletedatamodel.dat")
)
dwant = xgb.DMatrix(
    np.array(data_X), label=np.array(data_y), feature_names=feature_names
)
y_test_pred = bst.predict(dwant)
p450plant["scores"] = y_test_pred

In [ ]:
p450plant.to_pickle(our_data + "p450plant_deletedata.pkl")

In [ ]:
p450plant[(p450plant["enzyme"] == "CYP81A39")]

,enzyme,substrate,Binding,ESM1b,ECFP,scores
15642,CYP81A39,GEI,0.0,"[0.008906079456210136, 0.055766064673662186, 0...",0000000000000000000000000000000001000000000010...,0.002031
15643,CYP81A39,CAT,0.0,"[0.008906079456210136, 0.055766064673662186, 0...",0100000010000000000000000000010001001000000000...,0.018809
15644,CYP81A39,BAM,0.0,"[0.008906079456210136, 0.055766064673662186, 0...",0000000000000000000000000000000101001000000000...,0.152768
15645,CYP81A39,NME,0.0,"[0.008906079456210136, 0.055766064673662186, 0...",0001000000000000000000000100000001000000100000...,0.127032
15646,CYP81A39,DOC,0.0,"[0.008906079456210136, 0.055766064673662186, 0...",0100000010000000000000000000000001001000000000...,0.027965
...,...,...,...,...,...,...
15874,CYP81A39,COL,0.0,"[0.008906079456210136, 0.055766064673662186, 0...",0100000000000000000000000100000001001000000000...,0.084895
15875,CYP81A39,OME,0.0,"[0.008906079456210136, 0.055766064673662186, 0...",0000000000000000000001000000000001000000100000...,0.077338
15876,CYP81A39,GGE,0.0,"[0.008906079456210136, 0.055766064673662186, 0...",0000000000000000000000000000000001000000000010...,0.133735
15877,CYP81A39,biochaninA,0.0,"[0.008906079456210136, 0.055766064673662186, 0...",0000000000000000000000000000000001000000000000...,0.168791
